# 🗺️ 공공데이터 API 실습 2일차 - 시각화
**대상:** 빅데이터 과정 수강생  
**목표:** 1일차에서 가져온 데이터를 folium 지도 + pandas + matplotlib/plotly로 시각화한다

---
## 📋 오늘 실습 목록

| 번호 | 내용 | 라이브러리 |
|------|------|----------|
| 1 | 🗺️ 버스 정류장 지도에 표시 | folium |
| 2 | 🏥 병원 분포 지도 (클러스터링) | folium + MarkerCluster |
| 3 | 🌤️ 날씨 데이터 시각화 | matplotlib |
| 4 | 📊 음식점 업종 분석 | pandas + plotly |
| 5 | 🗺️ 나만의 지도 만들기 | folium (자유 실습) |

In [ ]:
# ✅ 필요한 라이브러리 설치 및 불러오기
# !pip install folium plotly

import requests
import json
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import matplotlib
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# 한글 폰트 설정
matplotlib.rcParams['font.family'] = 'Malgun Gothic'   # Windows
# matplotlib.rcParams['font.family'] = 'AppleGothic'  # Mac
matplotlib.rcParams['axes.unicode_minus'] = False

# 🔑 API 키
API_KEY = "여기에_API키_입력".strip()

# 대구 중심 좌표
DAEGU_LAT = 35.8714
DAEGU_LNG = 128.6014

print("설정 완료! ✅")

---
# 🗺️ 실습 1. 버스 정류장 지도에 표시하기

1일차에서 가져온 대구버스 정류장 데이터를 folium 지도에 올려봅시다!

In [ ]:
# 📡 대구버스 기초 데이터 가져오기 (1일차 복습)
url_bus = "https://apis.data.go.kr/6270000/dbmsapi02/getBasic02"
response_bus = requests.get(url_bus, params={"serviceKey": API_KEY})

if response_bus.status_code == 200:
    data_bus = response_bus.json()
    stops = data_bus["body"]["items"]["bs"]
    print(f"✅ 정류장 수: {len(stops)}개")
    print("샘플:", stops[0])
else:
    print("🔴 API 호출 실패:", response_bus.status_code)

In [ ]:
# 🗺️ folium 기본 지도 만들기
m1 = folium.Map(
    location=[DAEGU_LAT, DAEGU_LNG],  # 대구 중심
    zoom_start=12,                     # 줌 레벨 (1~18)
    tiles="OpenStreetMap"              # 지도 타일
)

# 정류장 마커 추가 (처음 100개만)
for stop in stops[:100]:
    folium.CircleMarker(
        location=[stop["yPos"], stop["xPos"]],  # 위도, 경도
        radius=5,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=0.6,
        popup=folium.Popup(stop["bsNm"], parse_html=True),  # 클릭시 정류장명
        tooltip=stop["bsNm"]  # 마우스 올리면 보임
    ).add_to(m1)

# 지도 저장 & 출력
m1.save("bus_stops_map.html")
print("✅ 지도 저장 완료: bus_stops_map.html")
m1  # Jupyter에서 바로 보기

In [ ]:
# 📌 MarkerCluster - 마커가 많을 때 군집으로 표시
m2 = folium.Map(location=[DAEGU_LAT, DAEGU_LNG], zoom_start=12)

# 클러스터 그룹 생성
cluster = MarkerCluster().add_to(m2)

# 전체 정류장 추가
for stop in stops:
    folium.Marker(
        location=[stop["yPos"], stop["xPos"]],
        popup=stop["bsNm"],
        icon=folium.Icon(color="blue", icon="bus", prefix="fa")
    ).add_to(cluster)

m2.save("bus_stops_cluster.html")
print(f"✅ 전체 {len(stops)}개 정류장 클러스터 지도 완성")
m2

In [ ]:
# 🎯 미션 1: 특정 이름이 포함된 정류장만 지도에 표시
keyword = "동대구"  # ← 원하는 키워드로 바꿔보세요

m3 = folium.Map(location=[DAEGU_LAT, DAEGU_LNG], zoom_start=13)

count = 0
for stop in stops:
    if keyword in stop["bsNm"]:
        folium.Marker(
            location=[stop["yPos"], stop["xPos"]],
            popup=f"{stop['bsNm']} ({stop['bsId']})",
            tooltip=stop["bsNm"],
            icon=folium.Icon(color="red", icon="info-sign")
        ).add_to(m3)
        count += 1

print(f"'{keyword}' 포함 정류장: {count}개")
m3

---
# 🏥 실습 2. 병원 분포 지도 (클러스터링)

대구 지역 병원 데이터를 가져와서 지도에 표시합니다.

In [ ]:
# 🏥 대구 병원 데이터 가져오기
url_hosp = "https://apis.data.go.kr/B551182/hospInfoServicev2/getHospBasisList"

all_hospitals = []

# 여러 페이지 가져오기
for page in range(1, 6):  # 5페이지 = 500개
    params = {
        "serviceKey": API_KEY,
        "numOfRows": "100",
        "pageNo": str(page),
        "sidoCd": "410000",  # 대구
        "_type": "json"
    }
    r = requests.get(url_hosp, params=params)
    if r.status_code == 200:
        items = r.json()["response"]["body"]["items"]["item"]
        all_hospitals.extend(items)
        print(f"  페이지 {page}: {len(items)}개 수집")

print(f"\n총 수집: {len(all_hospitals)}개 병원")

In [ ]:
# pandas DataFrame으로 변환
df_hosp = pd.DataFrame(all_hospitals)
print("컬럼:", df_hosp.columns.tolist())
print(df_hosp[["yadmNm", "clCdNm", "addr", "XPos", "YPos"]].head())

In [ ]:
# 🗺️ 병원 종류별 색상 지도
color_map = {
    "의원": "blue",
    "병원": "green",
    "종합병원": "red",
    "상급종합": "darkred",
    "요양병원": "orange"
}

m_hosp = folium.Map(location=[DAEGU_LAT, DAEGU_LNG], zoom_start=12)
cluster_hosp = MarkerCluster().add_to(m_hosp)

for _, row in df_hosp.iterrows():
    if pd.notna(row.get("YPos")) and pd.notna(row.get("XPos")):
        color = color_map.get(row.get("clCdNm", ""), "gray")
        folium.Marker(
            location=[float(row["YPos"]), float(row["XPos"])],
            popup=f"{row['yadmNm']} ({row.get('clCdNm','')})",
            tooltip=row["yadmNm"],
            icon=folium.Icon(color=color, icon="plus-sign")
        ).add_to(cluster_hosp)

m_hosp.save("hospitals_map.html")
print("✅ 병원 지도 완성")
m_hosp

In [ ]:
# 📊 병원 종류별 분포 차트
hosp_type_count = df_hosp["clCdNm"].value_counts().reset_index()
hosp_type_count.columns = ["종류", "수"]

fig = px.bar(
    hosp_type_count,
    x="종류", y="수",
    title="대구 병원 종류별 수",
    color="종류",
    text="수"
)
fig.update_traces(textposition="outside")
fig.show()

---
# 🌤️ 실습 3. 날씨 데이터 시각화

기상청 단기예보 데이터를 가져와서 오늘 날씨를 차트로 표현합니다.

In [ ]:
# 🌤️ 단기예보 - 오늘 시간대별 기온
now = datetime.now()
base_date = now.strftime("%Y%m%d")

# 단기예보는 0200, 0500, 0800, 1100, 1400, 1700, 2000, 2300 발표
base_times = ["0200", "0500", "0800", "1100", "1400", "1700", "2000", "2300"]
current_hour = now.hour

# 가장 최근 발표 시간 찾기
base_time = "0200"
for bt in base_times:
    if int(bt[:2]) <= current_hour:
        base_time = bt

url_fcst = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst"
params_fcst = {
    "serviceKey": API_KEY,
    "numOfRows": "300",
    "pageNo": "1",
    "dataType": "JSON",
    "base_date": base_date,
    "base_time": base_time,
    "nx": "89",  # 대구
    "ny": "90"
}

r_fcst = requests.get(url_fcst, params=params_fcst)
print("응답:", r_fcst.status_code)

if r_fcst.status_code == 200:
    items_fcst = r_fcst.json()["response"]["body"]["items"]["item"]
    df_fcst = pd.DataFrame(items_fcst)
    print(f"데이터 수: {len(df_fcst)}행")
    print("카테고리:", df_fcst["category"].unique())

In [ ]:
# 📈 오늘 시간대별 기온 그래프
df_temp = df_fcst[
    (df_fcst["category"] == "TMP") &
    (df_fcst["fcstDate"] == base_date)
].copy()

df_temp["fcstTime"] = df_temp["fcstTime"].astype(str).str.zfill(4)
df_temp["fcstValue"] = pd.to_numeric(df_temp["fcstValue"])
df_temp = df_temp.sort_values("fcstTime")

# matplotlib 그래프
plt.figure(figsize=(12, 5))
plt.plot(
    df_temp["fcstTime"],
    df_temp["fcstValue"],
    marker="o",
    linewidth=2,
    color="tomato",
    label="기온(℃)"
)
plt.fill_between(
    df_temp["fcstTime"],
    df_temp["fcstValue"],
    alpha=0.2,
    color="tomato"
)
for x, y in zip(df_temp["fcstTime"], df_temp["fcstValue"]):
    plt.annotate(f"{y}℃", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")

plt.title(f"대구 오늘({base_date}) 시간대별 기온", fontsize=14)
plt.xlabel("시간")
plt.ylabel("기온 (℃)")
plt.xticks(rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("today_temperature.png", dpi=150)
plt.show()
print("✅ 기온 그래프 저장: today_temperature.png")

In [ ]:
# 📊 강수확률 + 기온 함께 보기 (plotly)
df_pop = df_fcst[
    (df_fcst["category"] == "POP") &
    (df_fcst["fcstDate"] == base_date)
].copy()
df_pop["fcstValue"] = pd.to_numeric(df_pop["fcstValue"])
df_pop = df_pop.sort_values("fcstTime")

fig = go.Figure()

# 기온 라인
fig.add_trace(go.Scatter(
    x=df_temp["fcstTime"],
    y=df_temp["fcstValue"],
    name="기온(℃)",
    mode="lines+markers",
    line=dict(color="tomato", width=2),
    yaxis="y1"
))

# 강수확률 바
fig.add_trace(go.Bar(
    x=df_pop["fcstTime"],
    y=df_pop["fcstValue"],
    name="강수확률(%)",
    opacity=0.5,
    marker_color="steelblue",
    yaxis="y2"
))

fig.update_layout(
    title=f"대구 오늘({base_date}) 기온 & 강수확률",
    xaxis=dict(title="시간"),
    yaxis=dict(title="기온(℃)", side="left"),
    yaxis2=dict(title="강수확률(%)", side="right", overlaying="y", range=[0, 100]),
    legend=dict(x=0.01, y=0.99)
)
fig.show()

---
# 📊 실습 4. 음식점 데이터 분석

음식점 데이터를 pandas로 분석하고 plotly로 시각화합니다.

In [ ]:
# 🍽️ 음식점 데이터 샘플 생성 (API 미승인 시 대체용)
# ※ 실제 API 데이터가 있으면 아래 샘플 대신 사용하세요

import random
random.seed(42)

categories = ["한식", "중식", "일식", "양식", "분식", "치킨", "피자", "카페"]
districts = ["중구", "동구", "서구", "남구", "북구", "수성구", "달서구", "달성군"]

sample_data = [
    {
        "업종": random.choice(categories),
        "구": random.choice(districts),
        "위생상태": random.choice(["우수", "양호", "보통"]),
        "좌석수": random.randint(10, 100)
    }
    for _ in range(500)
]

df_food = pd.DataFrame(sample_data)
print("샘플 데이터 생성 완료:")
print(df_food.head())

In [ ]:
# 📊 업종별 음식점 수 - 파이차트
category_count = df_food["업종"].value_counts()

fig1 = px.pie(
    values=category_count.values,
    names=category_count.index,
    title="업종별 음식점 비율",
    hole=0.3
)
fig1.show()

In [ ]:
# 📊 구별 × 업종별 히트맵
pivot = df_food.groupby(["구", "업종"]).size().unstack(fill_value=0)

fig2 = px.imshow(
    pivot,
    title="구별 × 업종별 음식점 수 히트맵",
    color_continuous_scale="Blues",
    aspect="auto"
)
fig2.show()

In [ ]:
# 📊 구별 음식점 수 막대그래프
district_count = df_food["구"].value_counts().reset_index()
district_count.columns = ["구", "음식점수"]

fig3 = px.bar(
    district_count,
    x="구", y="음식점수",
    title="대구 구별 음식점 수",
    color="음식점수",
    color_continuous_scale="Viridis",
    text="음식점수"
)
fig3.update_traces(textposition="outside")
fig3.show()

In [ ]:
# 🎯 미션: 위생상태별 음식점 수를 구별로 비교하는 그래프 만들기
pivot2 = df_food.groupby(["구", "위생상태"]).size().reset_index(name="수")

fig4 = px.bar(
    pivot2,
    x="구", y="수",
    color="위생상태",
    title="구별 위생상태 분포",
    barmode="group",
    color_discrete_map={"우수": "#2ecc71", "양호": "#f39c12", "보통": "#e74c3c"}
)
fig4.show()

---
# 🗺️ 실습 5. 나만의 지도 만들기 (자유 실습)

folium 주요 기능을 배우고 직접 응용해봅니다!

In [ ]:
# 🗺️ folium 주요 기능 레퍼런스

m_ref = folium.Map(location=[DAEGU_LAT, DAEGU_LNG], zoom_start=13)

# 1. 기본 마커
folium.Marker(
    [35.8714, 128.6014],
    popup="대구시청",
    tooltip="여기 클릭!",
    icon=folium.Icon(color="red", icon="star")
).add_to(m_ref)

# 2. 원형 마커
folium.CircleMarker(
    [35.879, 128.627],
    radius=15,
    color="blue",
    fill=True,
    popup="동대구역"
).add_to(m_ref)

# 3. 원 (반경 표시)
folium.Circle(
    [35.8714, 128.6014],
    radius=1000,  # 1km 반경
    color="green",
    fill=False,
    popup="반경 1km"
).add_to(m_ref)

# 4. 폴리라인 (경로)
folium.PolyLine(
    [[35.8714, 128.6014], [35.879, 128.627]],
    color="orange",
    weight=3,
    opacity=0.8,
    popup="경로"
).add_to(m_ref)

# 5. 커스텀 팝업 (HTML)
popup_html = """
<div style='font-family: sans-serif; width: 150px;'>
    <h4>📍 동성로</h4>
    <p>대구의 대표 번화가</p>
</div>
"""
folium.Marker(
    [35.869, 128.596],
    popup=folium.Popup(popup_html, max_width=200),
    icon=folium.Icon(color="purple", icon="shopping-cart", prefix="fa")
).add_to(m_ref)

print("✅ folium 레퍼런스 지도 완성")
m_ref

In [ ]:
# 🏆 자유 실습: 나만의 지도 만들기
# 아이디어:
# - 내가 자주 가는 장소 지도
# - 대구 맛집 지도
# - API 데이터 + 지도 합치기
# - 병원 + 약국 같이 표시하기

my_map = folium.Map(location=[DAEGU_LAT, DAEGU_LNG], zoom_start=13)

# 여기에 나만의 지도를 만들어보세요!
# ...

my_map

---
# 📊 심화: 여러 API 데이터 합쳐서 분석하기

버스 정류장 근처 병원 찾기 - **두 API 데이터를 연결하는 실습!**

In [ ]:
# 📍 특정 정류장 반경 내 병원 찾기
import math

def distance_km(lat1, lng1, lat2, lng2):
    """두 좌표 간 거리 계산 (km)"""
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlng = math.radians(lng2 - lng1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlng/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# 동대구역 좌표
TARGET_LAT = 35.8793
TARGET_LNG = 128.6279
RADIUS_KM = 1.0  # 반경 1km

# 반경 내 병원 찾기
nearby_hospitals = []
for _, row in df_hosp.iterrows():
    try:
        dist = distance_km(TARGET_LAT, TARGET_LNG, float(row["YPos"]), float(row["XPos"]))
        if dist <= RADIUS_KM:
            row_dict = row.to_dict()
            row_dict["거리km"] = round(dist, 3)
            nearby_hospitals.append(row_dict)
    except:
        pass

df_nearby = pd.DataFrame(nearby_hospitals).sort_values("거리km")
print(f"동대구역 반경 {RADIUS_KM}km 내 병원: {len(df_nearby)}개")
print(df_nearby[["yadmNm", "clCdNm", "거리km"]].head(10))

In [ ]:
# 🗺️ 동대구역 반경 병원 지도
m_nearby = folium.Map(location=[TARGET_LAT, TARGET_LNG], zoom_start=15)

# 반경 원 표시
folium.Circle([TARGET_LAT, TARGET_LNG], radius=RADIUS_KM*1000,
              color="red", fill=True, fill_opacity=0.1, popup="반경 1km").add_to(m_nearby)

# 중심 마커
folium.Marker([TARGET_LAT, TARGET_LNG],
              popup="동대구역",
              icon=folium.Icon(color="red", icon="home")).add_to(m_nearby)

# 주변 병원
for _, row in df_nearby.iterrows():
    folium.Marker(
        [float(row["YPos"]), float(row["XPos"])],
        popup=f"{row['yadmNm']} ({row['거리km']}km)",
        tooltip=row["yadmNm"],
        icon=folium.Icon(color="blue", icon="plus-sign")
    ).add_to(m_nearby)

print(f"✅ 반경 내 병원 {len(df_nearby)}개 지도 완성")
m_nearby

---
# 📝 2일차 정리

| 기능 | 코드 | 용도 |
|------|------|------|
| 기본 지도 | `folium.Map(location=[lat,lng], zoom_start=12)` | 지도 생성 |
| 마커 | `folium.Marker([lat,lng], popup=...).add_to(m)` | 위치 표시 |
| 원형 마커 | `folium.CircleMarker(...)` | 데이터 크기 표현 |
| 반경 원 | `folium.Circle(radius=1000)` | 반경 표시 |
| 군집 | `MarkerCluster()` | 마커 많을 때 |
| 지도 저장 | `m.save("map.html")` | HTML로 저장 |
| 막대그래프 | `px.bar(df, x=..., y=...)` | 수량 비교 |
| 파이차트 | `px.pie(values=..., names=...)` | 비율 표시 |
| 히트맵 | `px.imshow(pivot)` | 2차원 분포 |

### 다음 시간 🔜
- 여러 API 데이터 pandas merge로 합치기
- Streamlit으로 웹 대시보드 만들기
- 미니 프로젝트: 내 주제로 전체 파이프라인 구현